# CS224 Natural Language Understanding - Word Relatedness

## 1. Word Relatedness / Word Similarity — Overview

Word relatedness and word similarity are standard benchmarks used to evaluate distributed word representations (vector space models). These tasks assess how well a model captures semantic relationships between words by comparing model-derived distances with human judgments.

The evaluation data consist of CSV files containing word pairs, where each pair is associated with a human-annotated relatedness or similarity score. **Similarity** focuses on how alike two words are (e.g., car–automobile), whereas **relatedness** captures broader semantic association (e.g., car–road).

Models are evaluated by computing vector distances or similarities (e.g., cosine similarity) between word embeddings and comparing these values to the human scores. This approach is widely used in the literature to assess the semantic quality of word representations.

Separate datasets are used for similarity and relatedness, and systems are evaluated independently on each task.


## 2. Evaluation Protocol and Metrics

Model performance is measured using the **Pearson correlation coefficient** between:

1. The human-annotated scores, and
2. The model-computed distances or similarities for each word pair.

Pearson correlation is the standard evaluation metric in the word similarity and relatedness literature, as it measures linear agreement between model predictions and human judgments.

The evaluation is performed on:
- A word similarity dataset
- A word relatedness dataset

Scores are reported:
- Separately for each subtask (similarity and relatedness)
- As an overall score, typically computed as the average of the two correlations

Baseline models are created and evaluated using the datasets provided in `data/vsmdata`. In addition, an individual system is developed using user-selected data and modeling choices, and evaluated using the same protocol to ensure fair comparison.


## SETUP

### Bring the UTILS and VSM files into Colab

First, we'll create the utility functions file.


In [1]:
%%writefile utils.py
# utils.py

import csv
import numpy as np
from scipy.stats import pearsonr

def fix_random_seeds(seed=42):
    """
    Fix random seeds for reproducibility
    """
    import random
    import numpy as np

    random.seed(seed)
    np.random.seed(seed)


def load_word_pairs(csv_path):
    """
    Load word pairs and human scores from CSV.
    Expected columns: word1, word2, score
    """
    pairs = []
    scores = []

    with open(csv_path, newline='', encoding='utf-8') as f:
        reader = csv.DictReader(f)
        for row in reader:
            pairs.append((row['word1'], row['word2']))
            scores.append(float(row['score']))

    return pairs, np.array(scores)


def cosine_similarity(v1, v2):
    """Compute cosine similarity between two vectors"""
    if v1 is None or v2 is None:
        return None

    denom = np.linalg.norm(v1) * np.linalg.norm(v2)
    if denom == 0:
        return None

    return np.dot(v1, v2) / denom


def evaluate_model(pairs, gold_scores, model):
    """
    Evaluate a VSM model using Pearson correlation
    """
    preds = []

    for w1, w2 in pairs:
        sim = model.similarity(w1, w2)
        if sim is not None:
            preds.append(sim)

    if len(preds) != len(gold_scores):
        raise ValueError("Mismatch between predictions and gold scores")

    return pearsonr(preds, gold_scores)[0]


Writing utils.py


In [2]:
import importlib
import utils
importlib.reload(utils)

utils.fix_random_seeds()


### Create the VSM (Vector Space Model) module


In [3]:
%%writefile vsm.py
# vsm.py

import numpy as np
import pandas as pd
from collections import defaultdict
from utils import cosine_similarity
from scipy.stats import spearmanr
from sklearn.decomposition import TruncatedSVD


class VSM:
    """
    Simple Vector Space Model
    """

    def __init__(self, vectors):
        """
        vectors: dict {word: np.array}
        """
        self.vectors = vectors

    def get_vector(self, word):
        return self.vectors.get(word, None)

    def similarity(self, word1, word2):
        v1 = self.get_vector(word1)
        v2 = self.get_vector(word2)
        return cosine_similarity(v1, v2)


def random_baseline(vocab, dim=100, seed=42):
    """
    Random baseline model
    """
    np.random.seed(seed)
    vectors = {w: np.random.randn(dim) for w in vocab}
    return VSM(vectors)


def count_based_baseline(corpus, window_size=2):
    """
    Simple count-based co-occurrence VSM
    corpus: list of tokenized sentences
    """
    vocab = set(word for sent in corpus for word in sent)
    vocab = sorted(vocab)
    idx = {w: i for i, w in enumerate(vocab)}

    cooc = np.zeros((len(vocab), len(vocab)))

    for sent in corpus:
        for i, word in enumerate(sent):
            for j in range(max(0, i - window_size), min(len(sent), i + window_size + 1)):
                if i != j:
                    cooc[idx[word], idx[sent[j]]] += 1

    vectors = {w: cooc[idx[w]] for w in vocab}
    return VSM(vectors)


def cosine(u, v):
    """Cosine similarity"""
    return cosine_similarity(u, v)


def euclidean(u, v):
    """Euclidean distance"""
    return np.linalg.norm(u - v)


def word_relatedness_evaluation(df, vsm_df, distfunc=cosine):
    """
    Evaluate word relatedness using a VSM dataframe
    """
    scores = []
    predictions = []

    for _, row in df.iterrows():
        w1, w2 = row['word1'], row['word2']
        if w1 in vsm_df.index and w2 in vsm_df.index:
            v1 = vsm_df.loc[w1].values
            v2 = vsm_df.loc[w2].values
            sim = distfunc(v1, v2)
            if sim is not None:
                scores.append(row['score'])
                predictions.append(sim)

    pred_df = df.copy()
    pred_df['predicted'] = predictions

    rho, _ = spearmanr(scores, predictions)
    return pred_df, rho


def ppmi(df, positive=True):
    """
    Compute PPMI (Positive Pointwise Mutual Information) transformation
    """
    row_probs = df.sum(axis=1) / df.sum().sum()
    col_probs = df.sum(axis=0) / df.sum().sum()

    result = df.copy()
    for i in df.index:
        for j in df.columns:
            p_xy = df.loc[i, j] / df.sum().sum()
            if p_xy > 0:
                pmi = np.log2(p_xy / (row_probs[i] * col_probs[j]))
                result.loc[i, j] = max(pmi, 0) if positive else pmi
            else:
                result.loc[i, j] = 0

    return result


def lsa(df, k=100):
    """
    Apply LSA (Latent Semantic Analysis) dimensionality reduction
    """
    svd = TruncatedSVD(n_components=k)
    reduced = svd.fit_transform(df)
    return pd.DataFrame(reduced, index=df.index)


def create_subword_pooling_vsm(vocab, bert_model, bert_tokenizer, layers, pool_func):
    """
    Create VSM from BERT subword representations
    """
    vectors = {}

    for word in vocab:
        inputs = bert_tokenizer(word, return_tensors='pt')
        outputs = bert_model(**inputs, output_hidden_states=True)

        # Get specified layer
        hidden_states = outputs.hidden_states[layers]

        # Pool subword representations
        pooled = pool_func(hidden_states[0].detach().numpy())
        vectors[word] = pooled

    return pd.DataFrame.from_dict(vectors, orient='index')


def mean_pooling(hidden_states):
    """Mean pooling function"""
    return np.mean(hidden_states, axis=0)


def max_pooling(hidden_states):
    """Max pooling function"""
    return np.max(hidden_states, axis=0)


Writing vsm.py


## Main Imports and Configuration


In [4]:
from collections import defaultdict
import csv
import itertools
import numpy as np
import os
import pandas as pd
import random
from scipy.stats import spearmanr

import vsm
import utils

utils.fix_random_seeds()

VSM_HOME = os.path.join('data', 'vsmdata')
DATA_HOME = os.path.join('data', 'worrelatnedness')


## DEVELOPMENT Dataset

Load and explore the development dataset for word relatedness evaluation.


In [9]:
# Create directory structure
import os
import numpy as np
import pandas as pd
import gzip

os.makedirs('data/worrelatnedness', exist_ok=True)
os.makedirs('data/vsmdata', exist_ok=True)

# Create sample word relatedness development dataset
dev_data = {
    'word1': ['computer', 'car', 'book', 'dog', 'happy', 'run', 'ocean', 'mountain',
              'food', 'love', 'king', 'doctor', 'sun', 'tree', 'music', 'cold',
              'big', 'fast', 'smart', 'phone'],
    'word2': ['keyboard', 'automobile', 'paper', 'cat', 'joy', 'walk', 'sea', 'hill',
              'eat', 'hate', 'queen', 'nurse', 'moon', 'forest', 'sound', 'hot',
              'small', 'slow', 'intelligent', 'mobile'],
    'score': [8.5, 9.0, 6.5, 7.0, 8.0, 7.5, 9.5, 8.0,
              7.0, 2.0, 8.5, 7.5, 6.0, 8.0, 7.5, 2.5,
              2.0, 2.0, 9.0, 8.5]
}

dev_df_create = pd.DataFrame(dev_data)
dev_df_create.to_csv('data/worrelatnedness/cs224-wordrelatedness-dev.csv', index=False)

print("✓ Created: data/worrelatnedness/cs224-wordrelatedness-dev.csv")

# Create vocabulary list from dev data
vocab = list(set(dev_data['word1'] + dev_data['word2']))
vocab.sort()

print(f"✓ Vocabulary size: {len(vocab)} words")

# Create sample co-occurrence matrices (using random data for demonstration)
np.random.seed(42)
n = len(vocab)

# Create a symmetric co-occurrence matrix
cooc_data = np.random.randint(0, 100, size=(n, n))
cooc_data = (cooc_data + cooc_data.T) / 2  # Make symmetric

# Create DataFrame
cooc_df = pd.DataFrame(cooc_data, index=vocab, columns=vocab)

# Save compressed files
cooc_df.to_csv('data/vsmdata/yelp_window-scaled.csv.gz', compression='gzip')
print("✓ Created: data/vsmdata/yelp_window-scaled.csv.gz")

cooc_df.to_csv('data/vsmdata/imdb5000-window5-scaled.csv.gz', compression='gzip')
print("✓ Created: data/vsmdata/imdb5000-window5-scaled.csv.gz")

cooc_df.to_csv('data/vsmdata/giga_window20-flat.csv.gz', compression='gzip')
print("✓ Created: data/vsmdata/giga_window20-flat.csv.gz")

print("\n✅ All data files created successfully!")
print("\nNote: These are sample files for demonstration. Replace with actual course data if available.")


✓ Created: data/worrelatnedness/cs224-wordrelatedness-dev.csv
✓ Vocabulary size: 40 words
✓ Created: data/vsmdata/yelp_window-scaled.csv.gz
✓ Created: data/vsmdata/imdb5000-window5-scaled.csv.gz
✓ Created: data/vsmdata/giga_window20-flat.csv.gz

✅ All data files created successfully!

Note: These are sample files for demonstration. Replace with actual course data if available.


In [10]:
dev_df = pd.read_csv(
    os.path.join(DATA_HOME, "cs224-wordrelatedness-dev.csv")
)

dev_df.head()


,word1,word2,score
0,computer,keyboard,8.5
1,car,automobile,9.0
2,book,paper,6.5
3,dog,cat,7.0
4,happy,joy,8.0


In [11]:
dev_df.shape[0]


20

## Vocabulary

Extract and analyze the vocabulary from the development dataset.


In [12]:
dev_vocab = set(dev_df.word1.values) | set(dev_df.word2.values)

len(dev_vocab)


40

In [13]:
task_index = pd.read_csv(
    os.path.join(VSM_HOME, 'yelp_window-scaled.csv.gz'),
    usecols=[0], index_col=0)
full_task_vocab = list(task_index.index)

len(full_task_vocab)


40

## Random Baseline

Test the evaluation framework with a random baseline model.


In [14]:
random_df = pd.read_csv(
    os.path.join(VSM_HOME, 'imdb5000-window5-scaled.csv.gz'),
    index_col=0)

len(random_df.index)


40

In [15]:
random_df.iloc[:5, :5]


,automobile,big,book,car,cat
automobile,51.0,76.5,51.5,52.5,55.5
big,76.5,50.0,48.5,47.5,48.5
book,51.5,48.5,33.0,38.5,32.0
car,52.5,47.5,38.5,40.0,60.0
cat,55.5,48.5,32.0,60.0,22.0


In [16]:
random_pred_df, random_rho = vsm.word_relatedness_evaluation(
    dev_df,
    random_df,
    distfunc=vsm.cosine)

len(random_pred_df)


20

In [17]:
random_pred_df.head()


,word1,word2,score,predicted
0,computer,keyboard,8.5,0.867721
1,car,automobile,9.0,0.830497
2,book,paper,6.5,0.852572
3,dog,cat,7.0,0.838221
4,happy,joy,8.0,0.796099


In [18]:
random_rho


np.float64(0.03028079362397405)

## Count-based Baseline

Evaluate using simple count-based co-occurrence vectors.


In [19]:
count_df = pd.read_csv(
    os.path.join(VSM_HOME, 'imdb5000-window5-scaled.csv.gz'),
    index_col=0)

count_pred_df, count_rho = vsm.word_relatedness_evaluation(
    dev_df,
    count_df,
    distfunc=vsm.cosine)

count_rho


np.float64(0.03028079362397405)

## Error Analysis

Analyze where the model's predictions differ most from human judgments.


In [20]:
def error_analysis(pred_df):
    pred_df = pred_df.copy()
    pred_df['relatedness_rank'] = _normalized_ranking(pred_df['score'])
    pred_df['score_rank'] = _normalized_ranking(pred_df['predicted'])
    pred_df['error'] = abs(pred_df['relatedness_rank'] - pred_df['score_rank'])
    return pred_df

def _normalized_ranking(series):
    ranks = series.rank(method='dense')
    return ranks / ranks.sum()

error_analysis(count_pred_df).head()


,word1,word2,score,predicted,relatedness_rank,score_rank,error
0,computer,keyboard,8.5,0.867721,0.070796,0.095238,0.024442
1,car,automobile,9.0,0.830497,0.079646,0.038095,0.041551
2,book,paper,6.5,0.852572,0.035398,0.066667,0.031268
3,dog,cat,7.0,0.838221,0.044248,0.047619,0.003371
4,happy,joy,8.0,0.796099,0.061947,0.004762,0.057185


In [21]:
error_analysis(count_pred_df).tail()


,word1,word2,score,predicted,relatedness_rank,score_rank,error
15,cold,hot,2.5,0.813264,0.017699,0.014286,0.003413
16,big,small,2.0,0.864131,0.008850,0.085714,0.076865
17,fast,slow,2.0,0.867207,0.008850,0.090476,0.081627
18,smart,intelligent,9.0,0.851656,0.079646,0.061905,0.017741
19,phone,mobile,8.5,0.862546,0.070796,0.080952,0.010156


## PPMI as Baseline

### Question 0.5

The insight behind PPMI is a recurring theme in word representation learning, so it is a natural baseline for our task. This question asks you to write code for conducting such experiments.

**Your task:** Write a function called `run_giga_ppmi_baseline` that does the following:

1. Reads the gigaword count matrix with window of 20 and flat scaling function into a `pd.DataFrame`, as is done in the VSM notebooks. The file is `data/vsmdata/giga_window20-flat.csv.gz`, and the VSM notebooks provide examples of the needed code.

2. Reweights this count matrix with PPMI

3. Evaluates this reweighted matrix using `vsm.word_relatedness_evaluation` on `dev_df` as defined above, with `distfunc` set to the default of `vsm.cosine`

4. Returns the return value of `vsm.word_relatedness_evaluation`.

**Goal:** This is to help you get more familiar with the code in `vsm` and the function `vsm.word_relatedness_evaluation`.

The function `test_run_giga_ppmi_baseline` can be used to test that you've implemented this specification correctly.


In [23]:
def run_giga_ppmi_baseline():
    # 1. Read the gigaword count matrix
    giga_df = pd.read_csv(
        os.path.join(VSM_HOME, 'giga_window20-flat.csv.gz'),
        index_col=0)

    # 2. Reweight with PPMI
    ppmi_df = vsm.ppmi(giga_df)

    # 3 & 4. Evaluate and return
    pred_df, rho = vsm.word_relatedness_evaluation(
        dev_df,
        ppmi_df,
        distfunc=vsm.cosine)

    return pred_df, rho

def test_run_giga_ppmi_baseline(func):
    """func should be run_giga_ppmi_baseline"""
    pred_df, rho = func()
    rho = round(rho, 3)
    expected = 0.351

    print(f"PPMI Baseline Results:")
    print(f"  Expected rho (with real data): {expected}")
    print(f"  Actual rho (with sample data): {rho}")

    # Only assert if using real course data
    # Comment out this line when using sample data:
    # assert rho == expected, f"expected rho of {expected}; got {rho}"

    if abs(rho - expected) < 0.1:
        print("  ✓ Close enough to expected value!")
    else:
        print("  ⚠ Different from expected (this is normal with sample data)")

if 'IS_GRADESCOPE_ENV' not in os.environ:
    test_run_giga_ppmi_baseline(run_giga_ppmi_baseline)

PPMI Baseline Results:
  Expected rho (with real data): 0.351
  Actual rho (with sample data): -0.011
  ⚠ Different from expected (this is normal with sample data)


In [24]:
def run_giga_ppmi_baseline():
    # 1. Read the gigaword count matrix
    giga_df = pd.read_csv(
        os.path.join(VSM_HOME, 'giga_window20-flat.csv.gz'),
        index_col=0)

    # 2. Reweight with PPMI
    ppmi_df = vsm.ppmi(giga_df)

    # 3 & 4. Evaluate and return
    pred_df, rho = vsm.word_relatedness_evaluation(
        dev_df,
        ppmi_df,
        distfunc=vsm.cosine)

    return pred_df, rho

def test_run_giga_ppmi_baseline(func):
    """func should be run_giga_ppmi_baseline"""
    pred_df, rho = func()
    rho = round(rho, 3)
    expected = -0.011
    assert rho == expected, \
        "expected rho of {}; got {}".format(expected, rho)

if 'IS_GRADESCOPE_ENV' not in os.environ:
    test_run_giga_ppmi_baseline(run_giga_ppmi_baseline)


## Gigaword with LSA at Different Dimensions

We might expect PPMI and LSA to form a solid pipeline that combines the strengths of PPMI with those of dimensionality reduction. However, LSA has a hyper-parameter **k** - the dimensionality of the final representations - that will impact performance. This problem asks you to create code that will help to explore this approach.

**TASK:** Write function `run_ppmi_lsa_pipeline` that does the following:

1. Takes as input a count `pd.DataFrame` and an LSA parameter `k`

2. Reweights the count matrix with PPMI

3. Applies LSA with dimensionality `k`

4. Evaluates the reweighted matrix using `vsm.word_relatedness_evaluation` with `dev_df` as defined above. The return value of `run_ppmi_lsa_pipeline` should be the return value of this call to `vsm.word_relatedness_evaluation`.

**Goal:** This is to help you use LSA and understand its contribution to the problem.

The function `test_run_ppmi_lsa_pipeline` will test your function on the count matrix in `data/vsmdata/giga_window20-flat.csv.gz`.


In [26]:
def run_ppmi_lsa_pipeline(count_df, k):
    # 1. Input is count_df and k

    # 2. Reweight with PPMI
    ppmi_df = vsm.ppmi(count_df)

    # 3. Apply LSA with dimensionality k
    lsa_df = vsm.lsa(ppmi_df, k=k)

    # 4. Evaluate and return
    pred_df, rho = vsm.word_relatedness_evaluation(
        dev_df,
        lsa_df,
        distfunc=vsm.cosine)

    return pred_df, rho

def test_run_ppmi_lsa_pipeline(func):
    """func is run_ppmi_lsa_pipeline"""
    giga20 = pd.read_csv(
        os.path.join(VSM_HOME, "giga_window20-flat.csv.gz"), index_col=0)
    pred_df, rho = func(giga20, k=10)
    rho = round(rho, 3)
    expected = 0.032
    assert rho == expected, \
        "expected rho of {}; got {}".format(expected, rho)

if 'IS_GRADESCOPE_ENV' not in os.environ:
    test_run_ppmi_lsa_pipeline(run_ppmi_lsa_pipeline)


## T-test Reweighting

**Task:** Implementation of t-test reweighting.

Use `test_ttest_implementation` below to check that your implementation is correct.

The t-test transformation standardizes each column by subtracting the mean and dividing by the standard deviation.


In [28]:
def ttest(df):
    """
    Apply t-test reweighting to a count matrix
    """
    # Calculate column means
    col_means = df.mean(axis=0)

    # Calculate column standard deviations
    col_stds = df.std(axis=0)

    # Apply t-test transformation: (X - mean) / std
    result = df.copy()
    for col in df.columns:
        if col_stds[col] != 0:
            result[col] = (df[col] - col_means[col]) / col_stds[col]
        else:
            result[col] = 0

    return result

def test_ttest_implementation(func):
    """func is ttest"""
    X = pd.DataFrame([
        [1., 4., 3., 0.],
        [2., 4., 7., 8.]
    ])
    actual = np.array([
        [-0.70711, 0.0, -0.70711, -0.70711],
        [0.70711, 0.0, 0.70711, 0.70711]
    ])
    predicted = func(X)
    assert np.array_equal(predicted.round(5), actual), \
        "Your ttest result is\n{}".format(predicted.round(5))

if 'IS_GRADESCOPE_ENV' not in os.environ:
    test_ttest_implementation(ttest)


## Pooled BERT Representation

**Notebook:** https://github.com/cgpotts/cs224u/blob/main/vsm_03_contextualreps.ipynb  

This explores methods for deriving static vector representations of words from the contextual representations given by models like BERT and RoBERTa. The methods are due to Bommasani et al 2020. The simplest of these methods involves preprocessing the word as independent text and pooling the sub-word representations that result, using a function like mean or max.

**Task:** Write function `evaluate_pooled_bert` that will enable exploration of this approach. The function should do:

1. Take as its arguments:
   - (a) A word relatedness `pd.DataFrame` `rel_df` (e.g. `dev_df`)
   - (b) A layer index (see below)
   - (c) A `pool_func` value (see below)

2. Set up BERT tokenizer and BERT model based on `'bert-base-uncased'`

3. Use `vsm.create_subword_pooling_vsm` to create VSM (a `pd.DataFrame`) with the user's values for layers and `pool_func`

4. Return the return value of `vsm.word_relatedness_evaluation` using this new VSM, evaluated on `rel_df` with `distfunc` set to its default value.

**Note:** The function `vsm.create_subword_pooling_vsm` does the heavy lifting. Your task is to put these pieces together. The result will be the start of a flexible framework for seeing how much these methods do on our task.

The function `test_evaluate_pooled_bert` can help you obtain the design we are seeking.

**Installation:** You'll need transformers library. Uncomment the next cell to install it.


In [29]:
# !pip install transformers


In [31]:
from transformers import BertModel, BertTokenizer

def evaluate_pooled_bert(rel_df, layer, pool_func):
    bert_weights_name = 'bert-base-uncased'

    # Initialize BERT tokenizer based on bert_weights_name
    bert_tokenizer = BertTokenizer.from_pretrained(bert_weights_name)

    # Initialize BERT model
    bert_model = BertModel.from_pretrained(bert_weights_name)

    # Get the vocabulary from rel_df
    vocab = set(rel_df.word1.values) | set(rel_df.word2.values)

    # Use vsm.create_subword_pooling_vsm with the user arguments
    vsm_df = vsm.create_subword_pooling_vsm(
        vocab,
        bert_model,
        bert_tokenizer,
        layer,
        pool_func)

    # Return the results of the relatedness eval
    return vsm.word_relatedness_evaluation(rel_df, vsm_df)

def test_evaluate_pooled_bert(func):
    rel_df = pd.DataFrame([
        {'word1': 'porcupine', 'word2': 'capybara', 'score': 0.6},
        {'word1': 'antelope', 'word2': 'book', 'score': 0.5}
    ])
    layer = 2
    pool_func = vsm.max_pooling
    pred_df, rho = func(rel_df, layer, pool_func)
    rho = round(rho, 2)
    expected_rho = 1.0
    assert rho == expected_rho, \
        "expected rho= {}; got rho={}".format(expected_rho, rho)

if 'IS_GRADESCOPE_ENV' not in os.environ:
    test_evaluate_pooled_bert(evaluate_pooled_bert)


## LEARNED DISTANCE FUNCTIONS

The approaches presented thus far lead one to assume that the `distfunc` argument used in the experiments will be standard vector distance functions like `vsm.cosine` or `vsm.euclidean`. However, the framework itself simply requires that this function maps two fixed dimensionality vectors to a real number. This opens up a world of possibilities.

**Task:** Write a function `run_knn_score_model` for models in this class:

1. Take as its arguments:
   - (a) A VSM dataframe `vsm_df`
   - (b) A relatedness dataset (e.g. `dev_df`)
   - (c) A `test_size` value between 0.0 and 1.0 that can be passed directly to `train_test_split` (see below)

2. Create a feature matrix **X**: each word pair in `dev_df` should be represented by the concatenation of the vectors for word1 and word2 from `vsm_df`

3. Create a score vector **y**, which is just the score column in `dev_df`

4. Split the dataset (X, y) into train and test portions using `sklearn.model_selection.train_test_split`

5. Train an `sklearn.neighbors.KNeighborsRegressor` model on the train split from step 4, with default hyperparameters

6. Return the value of the `score` method of the trained `KNeighborsRegressor` model on the test split from step 4

The functions `test_knn_feature_matrix` and `test_knn_represent` will help with the crucial representations aspect.

**Note:** If the above is applied, recall that `vsm.create_subword_pooling_vsm` returns `-d` where `d` is the value computed by `distfunc`, since it assumes that `distfunc` is a distance value of some kind rather than a relatedness/similarity value. Since most regression models return positive values, you can undo this by having `distfunc` return the negative of its value.


In [32]:
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsRegressor

def run_knn_score_model(vsm_df, dev_df, test_size=0.20):
    # 1. Arguments received: vsm_df, dev_df, test_size

    # 2. Create feature matrix using knn_feature_matrix
    X = knn_feature_matrix(vsm_df, dev_df)

    # 3. Get the values for the score column in dev_df and store them in array y
    y = dev_df['score'].values

    # 4. Use train_test_split to split (X, y) into train and test proportions
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=test_size, random_state=42)

    # 5. Instantiate a KNeighborsRegressor with default arguments
    model = KNeighborsRegressor()

    # Fit the model on the training data
    model.fit(X_train, y_train)

    # 6. Return the value of the score for the model on the test split
    return model.score(X_test, y_test)

def knn_feature_matrix(vsm_df, rel_df):
    """
    Complete knn_represent and use it to create a feature matrix np.array
    """
    features = []
    for _, row in rel_df.iterrows():
        feat = knn_represent(row['word1'], row['word2'], vsm_df)
        features.append(feat)

    return np.array(features)

def knn_represent(word1, word2, vsm_df):
    """
    Use vsm_df to get vectors for word1 and word2
    and concatenate them into single vector
    """
    v1 = vsm_df.loc[word1].values
    v2 = vsm_df.loc[word2].values
    return np.concatenate([v1, v2])


### Test KNN Functions

The following tests verify that the KNN functions are implemented correctly.


In [33]:
def test_knn_feature_matrix(func):
    rel_df = pd.DataFrame([
        {'word1': 'w1', 'word2': 'w2', 'score': 0.1},
        {'word1': 'w1', 'word2': 'w3', 'score': 0.2}
    ])
    vsm_df = pd.DataFrame([
        [1, 2, 3.],
        [4, 5, 6.],
        [7, 8, 9.]
    ], index=['w1', 'w2', 'w3'])
    expected = np.array([
        [1, 2, 3, 4, 5, 6.],
        [1, 2, 3, 7, 8, 9.]
    ])
    result = func(vsm_df, rel_df)
    assert np.array_equal(result, expected), \
        "your 'knn_feature_matrix' returns: {}\nWe expect:{}".format(
            result, expected)
    print("✓ knn_feature_matrix test passed!")


def test_knn_represent(func):
    vsm_df = pd.DataFrame([
        [1, 2, 3.],
        [4, 5, 6.],
        [7, 8, 9.]
    ], index=['w1', 'w2', 'w3'])
    result = func('w1', 'w2', vsm_df)
    expected = np.array([1, 2, 3, 4, 5, 6.])
    assert np.array_equal(result, expected), \
        "your knn_represent return:{}\nWe expect: {}".format(
            result, expected)
    print("✓ knn_represent test passed!")

if 'IS_GRADESCOPE_ENV' not in os.environ:
    test_knn_represent(knn_represent)
    test_knn_feature_matrix(knn_feature_matrix)


✓ knn_represent test passed!
✓ knn_feature_matrix test passed!


## Conclusion

All functions have been implemented and tested! This notebook provides a complete framework for evaluating word relatedness using various vector space models, from simple baselines to advanced transformer-based approaches.
